In [ ]:
import logging
from pathlib import Path

LOG_FMT = '%(asctime)s - %(name)s [%(threadName)s] %(funcName)s [%(levelname)s] %(message)s'
logging.basicConfig(level=logging.INFO, format=LOG_FMT)

validation_logger = logging.getLogger('validation')
#validation_logger.setLevel(logging.INFO)

file_handler = logging.FileHandler('validation-results.log', mode='a', encoding='utf-8')
file_handler.setFormatter(logging.Formatter(LOG_FMT))
validation_logger.addHandler(file_handler)

#validation_logger.info('Test')



import shared

import os
import time
import json

import polars as pl
import numpy as np

from sklearn.tree import DecisionTreeRegressor

import ngboost
from ngboost import NGBRegressor
from ngboost.distns import TFixedDf
from properscoring import crps_gaussian

from sklearn.metrics import mean_squared_error
from scipy import stats

from sklearn.metrics import r2_score

import seaborn as sns
import matplotlib.pyplot as plt


def env_flag(name: str, default: bool = False) -> bool:
    raw_value = os.getenv(name)
    if raw_value is None:
        return default
    return raw_value.strip().lower() in {'1', 'true', 'yes', 'y', 'on'}


def resolve_path(path_value: str | Path, base_dir: Path | None = None) -> Path:
    path = Path(path_value)
    if path.is_absolute():
        return path
    if base_dir is None:
        return path.resolve()
    return (base_dir / path).resolve()


def write_json_output(path_value: str | Path | None, payload: dict) -> None:
    if not path_value:
        return
    output_path = Path(path_value)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
np.random.seed(shared.random_seed())
f'Numpy {np.__version__}', f'ngboost {ngboost.__version__}'

In [ ]:
def standardize(series):
    """Standardize a pandas series"""
    mean = np.nanmean(series)
    std = np.nanstd(series)
    return (series - mean) / std, mean, std


#fixed_lambda = -0.50280368264246
#fixed_lambda = -0.2163132770134943

# Inverse transformation for the untransformed variable
def inverse_normalized_and_boxcox(normalized_bc_values, mean, sd, lambda_):
    # Reverse normalization
    data_unnormalized = normalized_bc_values * sd + mean
    # Reverse Box-Cox
    if lambda_ == 0:
        data_untransformed = np.exp(data_unnormalized)
    else:
       # Calculate the term inside the power operation
        inside = data_unnormalized * lambda_ + 1
        # Clip to a small positive value to avoid negatives/zero
        inside = np.clip(inside, 1e-6, None)
        data_untransformed = inside ** (1 / lambda_)
    
    return data_untransformed




In [ ]:
nb_start_time = time.time()

repo_root = resolve_path(os.getenv('JUKOLA_XML_MODEL_ROOT', '.'), Path.cwd())
data_dir = resolve_path(os.getenv('JUKOLA_DATA_DIR', repo_root / 'data'), repo_root)
optuna_results_dir = resolve_path(os.getenv('NGB_OPTUNA_RESULTS_DIR', repo_root / 'optuna-tuning-results'), repo_root)

full_data = env_flag('FULL_DATA', True)
batch_run_ts = os.getenv('BATCH_RUN_TS', 'not-set')

if batch_run_ts == 'not-set':
    os.environ.setdefault('RACE_TYPE', 've')
    os.environ.setdefault('FORECAST_YEAR', '2025')

# Use this to avoid overwriting
processing_batch_id = os.getenv('PROCESSING_BATCH_ID', 'ngboost-dev')
results_dir = resolve_path(
    os.getenv('NGB_RESULTS_DIR', str(repo_root / 'results' / processing_batch_id)),
    repo_root,
)
results_dir.mkdir(parents=True, exist_ok=True)

optuna_params_json_path = os.getenv('NGB_PARAMS_JSON', '').strip()
optuna_params_inline_json = os.getenv('NGB_PARAMS_INLINE_JSON', '').strip()
optuna_metrics_json_path = os.getenv('NGB_METRICS_JSON', '').strip()
optuna_trial_number = os.getenv('OPTUNA_TRIAL_NUMBER', '').strip()
enable_debug_plots = batch_run_ts == 'not-set' or env_flag('ENABLE_DEBUG_PLOTS', False)

logging.info(
    'Starting batch_run_ts=%s processing_batch_id=%s repo_root=%s data_dir=%s optuna_trial_number=%s',
    batch_run_ts,
    processing_batch_id,
    repo_root,
    data_dir,
    optuna_trial_number or '-',
)


In [ ]:
runs_path = resolve_path(
    os.getenv('LONG_RUNS_TSV', f'data/long_runs_and_running_order_{shared.race_id_str()}.tsv'),
    repo_root,
)
if not runs_path.exists():
    raise FileNotFoundError(f'Input TSV not found: {runs_path}')

logging.info('Loading runs_df from %s', runs_path)
runs_df = pl.read_csv(runs_path, separator='	')
runs_df


In [ ]:
runs_df.glimpse()

In [ ]:
example_name = 'oskari pirttikoski'
example_team = 'REAKTOR'

if shared.race_type() == 've':
    example_name = 'mari sane'
    example_team = 'VIILEÄT VENLAT'    

if shared.race_type() == 'ke':
    example_name = 'anna-liisa käppi'
    example_team = 'KENRAALI'    


In [ ]:
forecast_year = shared.forecast_year()
history_reference_df = runs_df.filter(pl.col("year") < forecast_year)

capped_paces = np.clip(runs_df["pace"].to_numpy(), a_min=4, a_max=40)
history_capped_paces = np.clip(history_reference_df["pace"].to_numpy(), a_min=4, a_max=40)

no_nans_history_capped_paces = history_capped_paces[~np.isnan(history_capped_paces)]
logging.info(
    f'history capped paces: {min(no_nans_history_capped_paces)} - {max(no_nans_history_capped_paces)}'
)

_, race_specific_bc_lambda = stats.boxcox(no_nans_history_capped_paces)
history_bc_transformed_paces = stats.boxcox(
    no_nans_history_capped_paces,
    lmbda=race_specific_bc_lambda,
)
_, bc_mean, bc_std = standardize(history_bc_transformed_paces)

bc_transformed_paces = stats.boxcox(capped_paces, lmbda=race_specific_bc_lambda)
normalized_bc_paces = (bc_transformed_paces - bc_mean) / bc_std

history_tc_values = history_reference_df["terrain_coefficient"].to_numpy()
_, tc_mean, tc_std = standardize(history_tc_values)
normalized_tc = (runs_df["terrain_coefficient"].to_numpy() - tc_mean) / tc_std
history_normalized_tc = (history_tc_values - tc_mean) / tc_std

logging.info(
    f'{shared.race_id_str()} {race_specific_bc_lambda=}, {bc_mean=}, {bc_std=}, {tc_mean=}, {tc_std=}'
)


def _inverse_normalized_and_boxcox(normalized_bc_values):
    return inverse_normalized_and_boxcox(normalized_bc_values, bc_mean, bc_std, race_specific_bc_lambda)



def _boxcox_and_normalize_out_of_sample(values):
    bc_transformed = stats.boxcox(values, lmbda=race_specific_bc_lambda)
    return (bc_transformed - bc_mean) / bc_std



def _safe_number(value, fallback):
    if value is None:
        return fallback
    if isinstance(value, (float, np.floating)) and np.isnan(value):
        return fallback
    return value



def build_past_country_features(feature_df: pl.DataFrame, min_country_runners: int = 50) -> pl.DataFrame:
    country_feature_frames = []
    year_values = sorted(feature_df["year"].drop_nulls().unique().to_list())

    for current_year in year_values:
        current_year_df = feature_df.filter(pl.col("year") == current_year).select([
            "row_id",
            "team_country",
        ])
        past_df = feature_df.filter(pl.col("year") < current_year)

        if past_df.height == 0:
            country_feature_frames.append(
                current_year_df.with_columns([
                    pl.lit("OTHER").alias("team_country_truncated"),
                    pl.lit(0.0).alias("c_bcp_median"),
                    pl.lit(1.0).alias("c_bcp_std"),
                    pl.lit(0).cast(pl.Int64).alias("c_num_runs"),
                    pl.lit(0).cast(pl.Int64).alias("c_n_unique_runners"),
                ])
            )
            continue

        country_bucket_df = (
            past_df
            .group_by("team_country")
            .agg(
                pl.col("unique_name").n_unique().alias("country_runner_count")
            )
            .with_columns(
                pl.when(pl.col("country_runner_count") > min_country_runners)
                .then(pl.col("team_country"))
                .otherwise(pl.lit("OTHER"))
                .alias("team_country_truncated")
            )
        )

        past_country_df = past_df.join(
            country_bucket_df.select([
                "team_country",
                "team_country_truncated",
                "country_runner_count",
            ]),
            on="team_country",
            how="left",
        )

        country_prior_df = (
            past_country_df
            .group_by("team_country_truncated")
            .agg([
                pl.col("tcn_bc_pace").median().alias("c_bcp_median"),
                pl.col("tcn_bc_pace").std().alias("c_bcp_std"),
                pl.col("tcn_bc_pace").count().alias("c_num_runs"),
                pl.col("unique_name").n_unique().alias("c_n_unique_runners"),
            ])
        )

        global_country_stats = past_df.select([
            pl.col("tcn_bc_pace").median().alias("c_bcp_median"),
            pl.col("tcn_bc_pace").std().alias("c_bcp_std"),
            pl.col("tcn_bc_pace").count().alias("c_num_runs"),
            pl.col("unique_name").n_unique().alias("c_n_unique_runners"),
        ]).to_dicts()[0]

        fallback_c_bcp_median = _safe_number(global_country_stats["c_bcp_median"], 0.0)
        fallback_c_bcp_std = _safe_number(global_country_stats["c_bcp_std"], 1.0)
        fallback_c_num_runs = int(_safe_number(global_country_stats["c_num_runs"], 0))
        fallback_c_n_unique_runners = int(_safe_number(global_country_stats["c_n_unique_runners"], 0))

        current_year_country_df = (
            current_year_df
            .join(
                country_bucket_df.select([
                    "team_country",
                    "team_country_truncated",
                ]),
                on="team_country",
                how="left",
            )
            .with_columns(
                pl.col("team_country_truncated").fill_null("OTHER")
            )
            .join(country_prior_df, on="team_country_truncated", how="left")
            .with_columns([
                pl.col("c_bcp_median").fill_null(pl.lit(fallback_c_bcp_median)),
                pl.col("c_bcp_std").fill_null(pl.lit(fallback_c_bcp_std)),
                pl.col("c_num_runs").fill_null(pl.lit(fallback_c_num_runs)),
                pl.col("c_n_unique_runners").fill_null(pl.lit(fallback_c_n_unique_runners)),
            ])
        )

        country_feature_frames.append(current_year_country_df)

    return pl.concat(country_feature_frames, how="vertical_relaxed")


lower, upper = np.percentile(history_normalized_tc, [10, 90])
history_tc_clamped = np.clip(history_normalized_tc, lower, upper)
tc_clamped = np.clip(normalized_tc, lower, upper)

history_uniques = np.sort(np.unique(history_tc_clamped))
history_gaps = np.diff(history_uniques)
history_gaps = history_gaps[history_gaps > 1e-8]
noise_scale = 0.5 * np.median(history_gaps) if history_gaps.size > 0 else 1e-6
logging.info(f'{shared.race_id_str()} uniform_tc {noise_scale=}')

history_noise = np.random.uniform(-noise_scale, noise_scale, size=history_tc_clamped.shape)
noise = np.random.uniform(-noise_scale, noise_scale, size=tc_clamped.shape)
history_tc_smooth = history_tc_clamped + history_noise
tc_smooth = tc_clamped + noise

from sklearn.preprocessing import QuantileTransformer

n_quantiles = max(1, min(100, history_tc_smooth.shape[0]))
qt = QuantileTransformer(
    output_distribution="uniform",
    n_quantiles=n_quantiles,
    random_state=shared.random_seed(),
)
qt.fit(history_tc_smooth.reshape(-1, 1))
tc_uni = qt.transform(tc_smooth.reshape(-1, 1)).ravel()

noise2 = np.random.uniform(-0.05, 0.05, size=tc_uni.shape)
tc_uni_final = tc_uni + noise2 + 0.5

noise_original = np.random.normal(loc=0.0, scale=0.3, size=normalized_tc.shape)

bc_df = runs_df.with_columns([
    pl.Series("bc_pace", normalized_bc_paces),
    pl.Series("normalized_tc", normalized_tc + noise_original),
    pl.Series("uniform_tc", tc_uni_final),
    (pl.col("marking_per_km") / history_reference_df["marking_per_km"].median()).alias("marking_norm"),
    (pl.col("vertical_per_km") / history_reference_df["vertical_per_km"].median()).alias("vertical_coef"),
    (pl.col("team_id") / pl.col("team_id").median().over("year") + 1).alias("normalized_team_id"),
    (pl.col("leg_dist") / pl.col("leg_dist").mean().over("year") + 1).alias("normalized_leg_dist"),
    (pl.col("run_num") / pl.col("run_num").median().over("year")).alias("run_num_norm"),
    (pl.col("run_num") <= 1).alias("first_time"),
    (pl.col("year") - pl.col("year").shift(1).rolling_mean(window_size=5, min_samples=1).over("unique_name")).alias("roll_5y_years_since_results"),
    (pl.col("pace") / pl.col("pace").median().over(["year", "leg"])).alias("pace_leg_ratio"),
]).fill_nan(None)

bc_df = bc_df.sort(["unique_name", "run_num"]).with_row_count("row_id")

bc_df = bc_df.with_columns([
    (pl.col("bc_pace") / pl.col("terrain_coefficient")).alias("tcn_bc_pace"),
    (pl.col("bc_pace") / pl.col("vertical_coef")).alias("vcn_bc_pace"),
])

bc_df = bc_df.with_columns([
    #(pl.col("tcn_bc_pace") * pl.col("normalized_team_id")).alias("tcn_bcp_ti_interaction"),
    #(pl.col("tcn_bc_pace") / pl.col("run_num")).alias("tcn_bcp_run_num_interaction"),
    (pl.col("normalized_team_id") / pl.col("run_num_norm")).alias("run_num_ti_interaction"),
])

bc_df = bc_df.with_columns([
    #Not in model
    pl.col("pace").shift(1).cumulative_eval(pl.element().drop_nulls().median()).over("unique_name").alias("history_pace_median"),
    
    pl.col("tcn_bc_pace").shift(1).cumulative_eval(pl.element().drop_nulls().median()).over("unique_name").alias("history_tcn_bcp_median"),

    pl.col("bc_pace").shift(1).rolling_mean(window_size=5, min_samples=1).over("unique_name").alias("roll_5y_bcp_mean"),
    
    pl.col("bc_pace").shift(1).rolling_median(window_size=15, min_samples=6).over("unique_name").alias("history_bcp_median"),
    
    pl.col("tcn_bc_pace").shift(1).rolling_mean(window_size=10, min_samples=1).over("unique_name").alias("roll_tcn_bcp_mean"),
    pl.col("vcn_bc_pace").shift(1).rolling_mean(window_size=10, min_samples=1).over("unique_name").alias("roll_vcn_bcp_mean"),
    pl.col("tcn_bc_pace").shift(1).rolling_mean(window_size=5, min_samples=1).over("unique_name").alias("roll_5y_tcn_bcp_mean"),
    #pl.col("tcn_bcp_ti_interaction").shift(1).rolling_mean(window_size=5, min_samples=1).over("unique_name").alias("roll_5y_tcn_bcp_ti_interaction_mean"),
    #pl.col("tcn_bcp_run_num_interaction").shift(1).rolling_mean(window_size=5, min_samples=1).over("unique_name").alias("roll_5y_tcn_bcp_run_num_interaction_mean"),
    pl.col("bc_pace").shift(1).rolling_std(window_size=5, min_samples=1).over("unique_name").alias("roll_5y_bcp_std"),
    pl.col("pace").shift(1).rolling_mean(window_size=5, min_samples=1).over("unique_name").alias("roll_5y_pace_mean"),
    pl.col("pace_leg_ratio").shift(1).rolling_mean(window_size=5, min_samples=1).over("unique_name").alias("roll_5y_pace_leg_ratio_mean"),
    pl.col("pace_leg_ratio").shift(1).rolling_std(window_size=10, min_samples=3).over("unique_name").alias("roll_pace_leg_ratio_std"),
    pl.col("bc_pace").shift(1).cumulative_eval(pl.element().std()).over("unique_name").alias("history_bcp_std"),
    pl.col("tcn_bc_pace").shift(1).cumulative_eval(pl.element().std()).over("unique_name").alias("history_tcn_bcp_std"),
    pl.col("tcn_bc_pace").shift(1).rolling_skew(window_size=15, min_samples=5).over("unique_name").alias("roll_tcn_bcp_skew"),
    pl.col("tcn_bc_pace").shift(1).rolling_kurtosis(window_size=15, min_samples=5).over("unique_name").alias("roll_tcn_bcp_kurtosis"),
    pl.col("pace").shift(1).rolling_skew(window_size=15, min_samples=5).over("unique_name").alias("roll_pace_skew"),
    (
        pl.col("tcn_bc_pace").shift(1).rolling_mean(10, min_samples=3).over("unique_name")
        - pl.col("tcn_bc_pace").shift(1).rolling_median(10, min_samples=3).over("unique_name")
    ).alias("roll_tcn_bcp_med_mean_diff"),
    (
        pl.col("tcn_bc_pace").shift(1).rolling_quantile(0.75, window_size=10, min_samples=4).over("unique_name")
        - pl.col("tcn_bc_pace").shift(1).rolling_quantile(0.25, window_size=10, min_samples=4).over("unique_name")
    ).alias("roll_tcn_bcp_iqr"),
    (
        pl.col("vcn_bc_pace").shift(1).rolling_quantile(0.75, window_size=10, min_samples=4).over("unique_name")
        - pl.col("vcn_bc_pace").shift(1).rolling_quantile(0.25, window_size=10, min_samples=4).over("unique_name")
    ).alias("roll_vcn_bcp_iqr"),
    pl.col("tcn_bc_pace").shift(1).rolling_std(window_size=5, min_samples=1).over("unique_name").alias("roll_5y_tcn_bcp_std"),
    pl.col("tcn_bc_pace").shift(1).rolling_std(window_size=10, min_samples=1).over("unique_name").alias("roll_tcn_bcp_std"),
    pl.col("vcn_bc_pace").shift(1).rolling_std(window_size=10, min_samples=1).over("unique_name").alias("roll_vcn_bcp_std"),
    pl.col("tcn_bc_pace").shift(1).rolling_min(window_size=10, min_samples=1).over("unique_name").alias("roll_tcn_bcp_min"),
    #pl.col("tcn_bc_pace").shift(1).rolling_max(window_size=10, min_samples=1).over("unique_name").alias("roll_tcn_bcp_max"),
    pl.col("terrain_coefficient").shift(1).rolling_mean(window_size=5, min_samples=1).over("unique_name").alias("roll_5y_tc_mean"),
    pl.col("vertical_coef").shift(1).rolling_mean(window_size=5, min_samples=1).over("unique_name").alias("roll_5y_vc_mean"),
    #pl.col("terrain_coefficient").shift(1).rolling_mean(window_size=10, min_samples=1).over("unique_name").alias("roll_tc_mean"),
    #pl.col("terrain_coefficient").shift(1).cumulative_eval(pl.element().drop_nulls().mean()).over("unique_name").alias("history_tc_mean"),
    #pl.col("normalized_team_id").shift(1).rolling_mean(window_size=5, min_samples=1).over("unique_name").alias("roll_5y_nti_mean"),
    #pl.col("normalized_leg_dist").shift(1).rolling_mean(window_size=5, min_samples=1).over("unique_name").alias("roll_5y_ndist_mean"),
])

country_feature_df = build_past_country_features(bc_df, min_country_runners=50)
bc_df = bc_df.join(
    country_feature_df.select([
        "row_id",
        "team_country_truncated",
        "c_bcp_median",
        "c_bcp_std",
        "c_num_runs",
        "c_n_unique_runners",
    ]),
    on="row_id",
    how="left",
)


team_group_cols = ["year", "team_id"]
history_col = "roll_5y_tcn_bcp_mean"

team_history_df = (
    bc_df
    .group_by(team_group_cols)
    .agg([
        pl.len().alias("team_members_total"),
        pl.col(history_col).count().alias("team_members_with_history"),
        pl.col(history_col).fill_null(0.0).sum().alias("team_history_sum"),
        (
            pl.col(history_col).fill_null(0.0) * pl.col(history_col).fill_null(0.0)
        ).sum().alias("team_history_sumsq"),
    ])
)

bc_df = (
    bc_df
    .join(team_history_df, on=team_group_cols, how="left")
    .with_columns([
        pl.col(history_col).fill_null(0.0).alias("own_history_value"),
        pl.col(history_col).is_not_null().cast(pl.Int64).alias("own_history_known"),
    ])
    .with_columns([
        (
            pl.col("team_members_with_history") - pl.col("own_history_known")
        ).alias("other_team_members_with_history"),
        (
            pl.col("team_history_sum") - pl.col("own_history_value")
        ).alias("other_team_history_sum"),
        (
            pl.col("team_history_sumsq")
            - pl.col("own_history_value") * pl.col("own_history_value")
        ).alias("other_team_history_sumsq"),
    ])
    .with_columns([
        pl.when(pl.col("other_team_members_with_history") > 0)
        .then(
            pl.col("other_team_history_sum")
            / pl.col("other_team_members_with_history")
        )
        .otherwise(None)
        .alias("other_team_members_roll_5y_tcn_bcp_mean"),

        pl.when(pl.col("other_team_members_with_history") > 1)
        .then(
            (
                pl.col("other_team_history_sumsq")
                - (
                    pl.col("other_team_history_sum")
                    * pl.col("other_team_history_sum")
                    / pl.col("other_team_members_with_history")
                )
            )
            / (pl.col("other_team_members_with_history") - 1)
        )
        .otherwise(None)
        .alias("other_team_members_roll_5y_tcn_bcp_var"),
    ])
    .with_columns([
        pl.when(pl.col("other_team_members_roll_5y_tcn_bcp_var").is_not_null())
        .then(pl.col("other_team_members_roll_5y_tcn_bcp_var").sqrt())
        .otherwise(None)
        .alias("other_team_members_roll_5y_tcn_bcp_std"),
    ])
    .drop([
        "own_history_value",
        "own_history_known",
        "team_history_sum",
        "team_history_sumsq",
        "other_team_history_sum",
        "other_team_history_sumsq",
        "other_team_members_roll_5y_tcn_bcp_var",
    ])
)


bc_df = bc_df.with_columns([
    #(pl.col("roll_5y_tcn_bcp_mean") / pl.col("history_tcn_bcp_median")).alias("roll_5y_history_median_ratio"),
    (pl.col("roll_5y_tcn_bcp_std") / pl.col("history_tcn_bcp_std")).alias("roll_5y_history_std_ratio"),
    (pl.col("history_bcp_std") - pl.col("history_tcn_bcp_std")).alias("history_tcn_bcp_std_diff"),
    (pl.col("roll_tcn_bcp_med_mean_diff") * pl.col("terrain_coefficient")).alias("roll_tcn_bcp_med_mean_diff_tc_interaction"),
    (pl.col("roll_vcn_bcp_mean") / pl.col("marking_norm")).alias("roll_vcn_bcp_mean_marking_interaction"),
    #(pl.col("roll_5y_tcn_bcp_mean") * pl.col("vertical_coef")).alias("roll_5y_tcn_bcp_vertical_interaction"),

    # Top feature
    (pl.col("roll_5y_pace_leg_ratio_mean") * pl.col("terrain_coefficient")).alias("roll_5y_pace_leg_ratio_tc_interaction"),

    
    (pl.col("roll_vcn_bcp_mean") * pl.col("vertical_coef")).alias("roll_vcn_bcp_mean_vc_interaction"),
    #(pl.col("normalized_team_id") * pl.col("roll_tcn_bcp_std")).alias("roll_tcn_bcp_std_mean_nti_interaction"),
    #(pl.col("normalized_team_id") * pl.col("roll_tcn_bcp_mean")).alias("roll_tcn_bcp_mean_nti_interaction"),
    (pl.col("normalized_team_id") * pl.col("c_bcp_median")).alias("c_bcp_median_nti_interaction"),
    (pl.col("normalized_team_id") * pl.col("c_bcp_std")).alias("c_bcp_std_nti_interaction"),
    #(pl.col("normalized_team_id") * pl.col("fn_scaled_pace")).alias("fn_scaled_pace_nti_interaction"),
    # (pl.col("terrain_coefficient") * pl.col("fn_scaled_pace") * pl.col("c_bcp_median")).alias("fn_scaled_pace_c_bcp_median_tc_interaction"),
    (pl.col("terrain_coefficient") * pl.col("fn_scaled_pace") ).alias("fn_scaled_pace_tc_interaction"),
    #(pl.col("vertical_coef") / pl.col("roll_5y_vc_mean") ).alias("vc_to_vc_history_interaction"),
    (pl.col("vertical_coef") * pl.col("fn_scaled_pace") * pl.col("c_bcp_median")).alias("fn_scaled_pace_c_bcp_median_vc_interaction"),
])

"""
cols_only_for_unknown_runners = [
    "c_bcp_median_nti_interaction",
    #"c_bcp_std_nti_interaction",
    "fn_scaled_pace_nti_interaction",
    "fn_scaled_pace_c_bcp_median_tc_interaction",
    "fn_scaled_pace_c_bcp_median_vertical_interaction",
]

bc_df = bc_df.with_columns([
    pl.when(pl.col("run_num") > 2)
      .then(None)
      .otherwise(pl.col(col))
      .alias(col)
    for col in cols_only_for_unknown_runners
])
"""


# Add dummy columns and keep the original 'leg' column
bc_df = bc_df.with_columns(pl.col("leg").cast(pl.Int64))

leg_dummies = bc_df.select("leg").to_dummies()
leg_dummy_cols = [col for col in leg_dummies.columns if col != "leg"]

bc_df = bc_df.hstack(leg_dummies.select(leg_dummy_cols))


feature_names = leg_dummy_cols + [
    "leg",
    "first_time",
    "run_num_norm",
    "roll_5y_years_since_results",
    "normalized_team_id",
    # "roll_tcn_bcp_mean_nti_interaction",
    "history_bcp_median",
    "roll_5y_tcn_bcp_mean",
    #"roll_5y_history_median_ratio",
    "history_tcn_bcp_std",
    "roll_5y_tcn_bcp_std",
    "history_tcn_bcp_std_diff",
    #"roll_tcn_bcp_std_mean_nti_interaction",
    "roll_tcn_bcp_kurtosis",
    "roll_tcn_bcp_skew",
    "roll_tcn_bcp_med_mean_diff",
    "roll_tcn_bcp_iqr",
    "roll_vcn_bcp_iqr", 
    "roll_tcn_bcp_med_mean_diff_tc_interaction",
    "roll_5y_history_std_ratio",
    "roll_tcn_bcp_min",
    #"roll_tcn_bcp_max",
    #"roll_vcn_bcp_mean",
    "roll_vcn_bcp_mean_vc_interaction",
    "roll_vcn_bcp_std",
    "roll_5y_pace_leg_ratio_mean",
    "roll_pace_leg_ratio_std",

    #"roll_5y_pace_leg_ratio_tc_interaction",
    # "uniform_tc",

    #"roll_5y_tcn_bcp_ti_interaction_mean",
    #"roll_5y_tcn_bcp_run_num_interaction_mean",
    "roll_5y_tc_mean",
    #"roll_5y_vc_mean",
    "roll_vcn_bcp_mean_marking_interaction",
    #"roll_5y_tcn_bcp_vertical_interaction",
    "fn_scaled_pace",
    #"c_bcp_median",
    #"c_bcp_std",
    "c_bcp_median_nti_interaction",
    # "c_bcp_std_nti_interaction",
    #"fn_scaled_pace_nti_interaction",
    #"fn_scaled_pace_c_bcp_median_tc_interaction",
    "fn_scaled_pace_tc_interaction",
    "fn_scaled_pace_c_bcp_median_vc_interaction",

    'other_team_members_roll_5y_tcn_bcp_mean',
    'other_team_members_roll_5y_tcn_bcp_std',
    #'team_members_with_history',
    'other_team_members_with_history',
    #'team_members_total',
    
]

leaky_until_rebuilt = []

first_pass_drops = [
    #"roll_5y_history_median_ratio",
    "roll_5y_history_std_ratio",
    #"roll_tcn_bcp_skew",
    "roll_tcn_bcp_kurtosis",
    "roll_tcn_bcp_iqr",
    "roll_tcn_bcp_med_mean_diff",
    "roll_tcn_bcp_med_mean_diff_tc_interaction",
    "history_tcn_bcp_std",
    "history_tcn_bcp_std_diff",
    "roll_vcn_bcp_std",
    "roll_pace_leg_ratio_std",
]

feature_names = [
    name for name in feature_names
    if name not in leaky_until_rebuilt + first_pass_drops
]


In [ ]:
column_name = 'roll_tcn_bcp_skew'

logging.info(f'{column_name} stats:\n{bc_df[column_name].describe()}')
plt.figure(figsize=(14, 10))

ax = sns.kdeplot(bc_df, x=column_name, hue='year',  palette='bright') #clip=[3, 30],  


In [ ]:
 # Filter the dataframe for the year 2024
df_2024 = bc_df.filter(pl.col('year') == 2024)

# Print the statistics for 2024 only
logging.info(f'{column_name} stats for 2024:\n{df_2024[column_name].describe()}')

# Plot the distribution for 2024 only
plt.figure(figsize=(14, 10))
ax = sns.kdeplot(data=df_2024.to_pandas(), x=column_name)
plt.title(f'Distribution of {column_name} in 2024')
plt.show()


In [ ]:
bc_df.glimpse()

In [ ]:
bc_df.filter(
    pl.col('unique_name').str.contains(example_name)
)#.select(
#    ['unique_name', 'year', 'run_num', 'pace', 'bc_pace', 'terrain_coefficient', 'vertical_per_km', 'history_pace_median',  'roll_bcp_median',  'roll_5y_pace_mean',   'roll_5y_bcp_median', 'roll_5y_bcp_mean', 'roll_5y_bcp_std', 'roll_5y_tc_mean', 'roll_bcp_min', 'roll_bcp_max', ]
#)
#example_bc_df

In [ ]:

#plt.figure(figsize=(14, 10))
#sns.lmplot(example_bc_df, y='bc_pace', x='roll_5y_history_median_ratio')

In [ ]:
#features_df = df
#features_df = bc_df
history_df = bc_df.filter(
    # TODO filter by bc_pace and consider also nan    history_df.filter(history_df["bc_pace"].is_nan())
    pl.col('pace').is_not_null() & (pl.col('year') < shared.forecast_year())  & (pl.col('year') >= 2004) # Quick hack to Reduce num of no history 
)
history_df.filter(
    pl.col('unique_name').str.contains(example_name)
)


In [ ]:

running_order_df = bc_df.filter(
    pl.col('year') == shared.forecast_year()
)
running_order_df.filter(
    pl.col('team').str.contains(example_team)
)
running_order_df

In [ ]:

logging.info(f'Extremes:\n{history_df["pace"].describe(percentiles=[0.0001, 0.001, 0.999, 0.9999])}')
logging.info(f'BoxCox Extremes:\n{history_df["bc_pace"].describe(percentiles=[0.0001, 0.001, 0.999, 0.9999])}')
history_df.sort('pace')
#sum(history_df["bc_pace"].is_null())
#history_df.filter(history_df["bc_pace"].is_nan())
#sum(history_df["bc_pace"].is_nan())

In [ ]:

# Create a new column in history_df that combines 'year' and 'leg' as a string
history_df = history_df.with_columns(
    pl.concat_str(["year", "leg"], separator="_").alias("year_leg_str")
)

recent_unique_year_legs = history_df.filter(
    pl.col('year') > shared.forecast_year() - 16
).select("year_leg_str").unique().sort("year_leg_str")

# unique values without replacement using a fixed random seed
#validation_and_test_set_leg_count = shared.num_legs[shared.race_type()] * 2
validation_and_test_set_leg_count = 15
if shared.race_type() == 'ke':
    validation_and_test_set_leg_count = 1
sampled_unique = recent_unique_year_legs.sample(n=validation_and_test_set_leg_count, with_replacement=False, seed=shared.random_seed())

# Convert the sampled column to a Python list
vat_random_year_legs = sorted(sampled_unique["year_leg_str"].to_list())



# Split the DataFrame into two:
validation_and_test_df = history_df.filter(pl.col("year_leg_str").is_in(vat_random_year_legs))
training_df = history_df.filter(~pl.col("year_leg_str").is_in(vat_random_year_legs))
vat_random_year_legs

In [ ]:
history_df = history_df.filter(pl.col("year") >= 2000)

In [ ]:

#print(f'Available feature cols: {history_df.columns}')


X_history = history_df.select(feature_names).to_numpy()

X_train = training_df.select(feature_names).to_numpy()
X_vat = validation_and_test_df.select(feature_names).to_numpy()
X_forecast_year = running_order_df.select(feature_names).to_numpy()

# Select target column and flatten the array
y_train = training_df.select('bc_pace').to_numpy().flatten()
y_vat = validation_and_test_df.select('bc_pace').to_numpy().flatten()
# Should we if here ?

# Count the non-null values in 'bc_pace'
fy_non_null_count = running_order_df["bc_pace"].is_not_null().sum()
y_forecast_year = []
if fy_non_null_count > 0:
    y_forecast_year = running_order_df.select('bc_pace').to_numpy().flatten()

X_train.shape, y_train.shape, X_vat.shape, y_vat.shape, X_forecast_year.shape

In [ ]:
if batch_run_ts == 'not-set':
    corr_matrix = history_df.select(feature_names).to_pandas().corr()
    #corr_matrix
    plt.figure(figsize=(14, 10))
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
    display(corr_matrix)


In [ ]:
if batch_run_ts == "not-set" or True:
    feature_df = history_df.select(feature_names).to_pandas()
    corr_matrix = feature_df.corr(numeric_only=True)

    abs_corr = corr_matrix.abs().copy()
    np.fill_diagonal(abs_corr.values, 0.0)

    strongest_pairs = (
        abs_corr.where(np.triu(np.ones(abs_corr.shape), k=1).astype(bool))
        .stack()
        .sort_values(ascending=False)
        .head(10)
        .rename("abs_corr")
        .reset_index()
        .rename(columns={"level_0": "feature_a", "level_1": "feature_b"})
    )

    top_feature_names = sorted(
        set(strongest_pairs["feature_a"]).union(strongest_pairs["feature_b"])
    )

    top_corr_matrix = corr_matrix.loc[top_feature_names, top_feature_names]

    plt.figure(figsize=(12, 9))
    sns.heatmap(
        top_corr_matrix,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
    )
    plt.title("Features appearing in the top 10 correlated pairs")
    plt.tight_layout()

    display(strongest_pairs)
    display(top_corr_matrix)

In [ ]:
import numpy as np
from scipy.stats import t as scipy_t
from ngboost.distns.t import TFixedDf, TFixedDfLogScore


def make_t_fixed_df(df_value: float):
    class TFixedDfCustom(TFixedDf):
        fixed_df = float(df_value)
        scores = [TFixedDfLogScore]

        @staticmethod
        def fit(y_true):
            _, location, scale = scipy_t.fit(y_true, fdf=float(df_value))
            return np.array([location, np.log(scale)])

    TFixedDfCustom.__name__ = f"TFixedDf_{str(df_value).replace('.', '_')}"
    return TFixedDfCustom


TFixedDf8 = make_t_fixed_df(8.0)
TFixedDf6 = make_t_fixed_df(6.0)
TFixedDf5 = make_t_fixed_df(5.0)
TFixedDf4 = make_t_fixed_df(4.0)

In [ ]:
default_best_params_paths = {
    ('ju', 2025): 'best-params/race_type=ju/study=ju-student-t-df4-v2/params.json',
    #('ju', 2025): 'best-params/race_type=ju/study=ju-student-t-df3-v1/params.json',
    # ('ju', 2025): '/Users/oskari/koodi/jukola-ngboost/optuna-tuning-results/race_id=ju_fy_2025/v26_ju_fy_2025-t.113-20250610_233255-best-params.json',
    #  
    ('ve', 2025): 'best-params/race_type=ve/study=ve-student-t-df4-v1/params.json',
    #('ve', 2025): '/Users/oskari/koodi/jukola-ngboost/optuna-tuning-results/race_id=ve_fy_2025/v26_ve_fy_2025-t.248-20250612_234555-best-params.json',
}


def load_best_params() -> tuple[dict, Path | None, str]:
    if optuna_params_inline_json:
        return json.loads(optuna_params_inline_json), None, 'NGB_PARAMS_INLINE_JSON'

    default_best_params_path = default_best_params_paths.get((shared.race_type(), shared.forecast_year()))
    if default_best_params_path is None:
        default_best_params_path = default_best_params_paths.get((shared.race_type(), 2025))

    selected_path_value = optuna_params_json_path or default_best_params_path
    if not selected_path_value:
        raise FileNotFoundError(
            f'No best params path configured for race_type={shared.race_type()} forecast_year={shared.forecast_year()}'
        )

    selected_path = resolve_path(selected_path_value, repo_root)
    if not selected_path.exists():
        raise FileNotFoundError(f'Best params JSON not found: {selected_path}')

    with open(selected_path, 'r', encoding='utf-8') as f:
        loaded_best_params = json.load(f)

    selected_source = 'NGB_PARAMS_JSON' if optuna_params_json_path else 'default_best_params_path'
    return loaded_best_params, selected_path, selected_source


best_params, best_params_path, selected_params_source = load_best_params()
logging.info(
    'Loaded hyperparameters from %s source=%s',
    best_params_path if best_params_path else '<inline-json>',
    selected_params_source,
)

# Separate the parameters into two dictionaries:
# 1. For the tree (parameters with the "Base__" prefix)
# 2. For the NGBoost model itself
base_params = {}
ngb_params = {}

for key, value in best_params.items():
    if key.startswith("Base__"):
        # Remove the prefix to match the DecisionTreeRegressor arguments.
        base_params[key.replace("Base__", "")] = value
    else:
        ngb_params[key] = value

learning_rate = ngb_params.get('learning_rate', 0.01)
extra_iterations = int(os.getenv('NGB_EXTRA_ITERATIONS', '50'))
extra_learning_rate = float(os.getenv('NGB_EXTRA_LEARNING_RATE', '0.0'))

min_samples_leaf = base_params.get('min_samples_leaf', 125)
min_samples_split = 2 * min_samples_leaf

# Create the decision tree using the tuned hyperparameters.
tree_learner = DecisionTreeRegressor(
    criterion='friedman_mse',
    min_samples_split=min_samples_split,
    min_samples_leaf=min_samples_leaf,
    max_features=base_params.get("max_features", 0.6),
    #max_features=0.4,
    max_depth=base_params.get("max_depth", 3),
    #max_depth=30,
    splitter="best",
    random_state=shared.random_seed(),  # your shared seed function
)

"""
from sklearn.ensemble import HistGradientBoostingRegressor

base_hgbr = HistGradientBoostingRegressor(
    max_iter=100,
    #learning_rate=learning_rate,
    max_depth=base_params.get("max_depth", 3),
    min_samples_leaf=min_samples_leaf,
    random_state=shared.random_seed(),
    max_features=base_params.get("max_features", 0.6),
    verbose=0,
)
"""

what_was_changed = os.getenv(
    'MODEL_NOTE',
    'TFixedDf, TFixedDf4, env extra 500, TFixedDf8, TFixedDf, skew+, disable more, no run_num, reduced vc, vcn+ <-2026',
)

used_hyperparameters = {
    'Base__max_depth': int(base_params.get('max_depth', 3)),
    'Base__max_features': float(base_params.get('max_features', 0.6)),
    'Base__min_samples_leaf': int(min_samples_leaf),
    'Base__min_samples_split': int(min_samples_split),
    'col_sample': float(ngb_params.get('col_sample', 0.6)),
    'early_stopping_rounds': 10,
    'learning_rate': float(learning_rate + extra_learning_rate),
    'minibatch_frac': float(ngb_params.get('minibatch_frac', 0.7)),
    'n_estimators': int(ngb_params.get('n_estimators', 100) + extra_iterations),
    'natural_gradient': True,
    'random_state': int(shared.random_seed()),
}

from ngboost.distns import TFixedDf
#from ngboost.distns import T
   
 
# Create the NGBoost model using the tuned hyperparameters.
ngb = NGBRegressor(
    #Dist=Normal,
    Dist=TFixedDf,
    # Dist=T, # crashes
    # Dist=BoundedT,
    #Dist=TFixedDf4,
    #Score=scores.CRPScore,
    Base=tree_learner,
    #Base=base_hgbr,
    early_stopping_rounds=used_hyperparameters['early_stopping_rounds'],
    n_estimators=used_hyperparameters['n_estimators'],
    learning_rate=used_hyperparameters['learning_rate'],
    minibatch_frac=used_hyperparameters['minibatch_frac'],
    col_sample=used_hyperparameters['col_sample'],
    natural_gradient=True,
    # natural_gradient=False,
    random_state=shared.random_seed(),
    verbose=True,
)



In [ ]:
race_id_str = shared.race_id_str()

logging.info(f'{race_id_str} {batch_run_ts} Starting with training data: {X_train.shape=}, {what_was_changed=}')
start_time = time.time()
# ngb.fit(X_train, y_train, X_val=X_val, Y_val=y_val)
ngb.fit(X_train, y_train, X_val=X_vat, Y_val=y_vat)
duration = time.time() - start_time
logging.info(f'{race_id_str} {batch_run_ts} Fitting took {duration:.1f} secs')

In [ ]:
print(f'{race_id_str} {batch_run_ts} {ngb.best_val_loss_itr=}')
validation_logger.info(f"{race_id_str} {batch_run_ts} {ngb.best_val_loss_itr=}, PARAMS: {ngb.get_params()}, FEATURES: {len(feature_names)} names: {sorted(feature_names)}")
model_params = ngb.get_params()

In [ ]:
# Step 6: Make predictions
y_vat_pred = ngb.predict(X_vat, max_iter=ngb.best_val_loss_itr)
y_vat_dist = ngb.pred_dist(X_vat, max_iter=ngb.best_val_loss_itr)

# Compute CRPS for each observation:
crps_values = crps_gaussian(y_vat, y_vat_dist.loc, y_vat_dist.scale)
average_crps = float(np.mean(crps_values))

# Compute Negative Log-Likelihood (NLL) for each observation:
# The pred_dists object provides a logpdf method, which returns the log likelihood of each true value.
nll_values = -y_vat_dist.logpdf(y_vat)
average_nll = float(np.mean(nll_values))

validation_r2 = float(r2_score(y_vat, y_vat_pred))
validation_logger.info(
    f"{race_id_str} {batch_run_ts} Validation Set R² Score: {validation_r2:.3f}, CRPS: {average_crps:.3f}, NLL: {average_nll:.3f} # {what_was_changed}"
)

y_forecast_year_pred = ngb.predict(X_forecast_year, max_iter=ngb.best_val_loss_itr)
forecast_metrics = None
y_forecast_year_dist = None
if len(y_forecast_year):
    y_forecast_year_dist = ngb.pred_dist(X_forecast_year, max_iter=ngb.best_val_loss_itr)
    forecast_mse = float(mean_squared_error(y_forecast_year, y_forecast_year_pred))
    forecast_r2 = float(r2_score(y_forecast_year, y_forecast_year_pred))
    forecast_crps = float(np.mean(crps_gaussian(y_forecast_year, y_forecast_year_dist.loc, y_forecast_year_dist.scale)))
    forecast_nll = float(np.mean(-y_forecast_year_dist.logpdf(y_forecast_year)))
    forecast_metrics = {
        'crps': forecast_crps,
        'mse': forecast_mse,
        'nll': forecast_nll,
        'r2': forecast_r2,
        'rows': int(len(y_forecast_year)),
    }
    print(f'{race_id_str} Forecast year (Test Set) Mean Squared Error: {forecast_mse:.3f}')
    logging.info(
        f"{race_id_str} Forecast year (Test Set) R² Score: {forecast_r2:.3f}, CRPS: {forecast_crps:.3f}, NLL: {forecast_nll:.3f} # {what_was_changed}"
    )

training_metrics = {
    'batch_run_ts': batch_run_ts,
    'best_params_path': str(best_params_path) if best_params_path else None,
    'best_val_loss_itr': int(ngb.best_val_loss_itr),
    'feature_count': int(len(feature_names)),
    'feature_names': sorted(feature_names),
    'fit_duration_seconds': float(duration),
    'forecast_year': int(shared.forecast_year()),
    'forecast_year_metrics': forecast_metrics,
    'optuna_trial_number': int(optuna_trial_number) if optuna_trial_number else None,
    'processing_batch_id': processing_batch_id,
    'race_id': race_id_str,
    'race_type': shared.race_type(),
    'selected_params_source': selected_params_source,
    'training_rows': int(len(y_train)),
    'used_hyperparameters': used_hyperparameters,
    'validation_metrics': {
        'crps': average_crps,
        'nll': average_nll,
        'r2': validation_r2,
        'rows': int(len(y_vat)),
    },
    'what_was_changed': what_was_changed,
}

ngboost_metrics_path = results_dir / f'ngboost_metrics_{race_id_str}.json'
write_json_output(ngboost_metrics_path, training_metrics)
write_json_output(optuna_metrics_json_path, training_metrics)
logging.info('Wrote NGBoost metrics to %s', ngboost_metrics_path)


In [ ]:
if enable_debug_plots and ngb.feature_importances_ is not None:

    pred_dist_vat = ngb.pred_dist(X_vat)
    pred_df = np.asarray(pred_dist_vat.df)
    
    logging.info(
        "pred_df quantiles=%s",
        {
            "min": float(np.min(pred_df)),
            "p10": float(np.quantile(pred_df, 0.10)),
            "p25": float(np.quantile(pred_df, 0.25)),
            "p50": float(np.quantile(pred_df, 0.50)),
            "p75": float(np.quantile(pred_df, 0.75)),
            "p90": float(np.quantile(pred_df, 0.90)),
            "max": float(np.max(pred_df)),
        },
    )

In [ ]:
if enable_debug_plots and ngb.feature_importances_ is not None:

    ## Feature importance for loc trees
    feature_importance_loc = ngb.feature_importances_[0]
    df_loc = pl.DataFrame({'feature': feature_names, 'importance': feature_importance_loc}).sort('importance')

    ## Feature importance for scale trees
    feature_importance_scale = ngb.feature_importances_[1]
    df_scale = pl.DataFrame({'feature': feature_names, 'importance': feature_importance_scale}).sort('importance')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
    fig.suptitle(f'Feature importance plot for distribution parameters {race_id_str}', fontsize=17)
    sns.barplot(x='importance', y='feature', ax=ax1, data=df_loc, color='skyblue').set_title('loc param')
    sns.barplot(x='importance', y='feature', ax=ax2, data=df_scale, color='skyblue').set_title('scale param')


In [ ]:
def _print_shap(max_rows=400, approximate=True):
    import shap

    shap.initjs()

    x_source = X_vat if len(X_vat) else X_train
    sample_size = min(max_rows, len(x_source))
    rng = np.random.default_rng(shared.random_seed())
    sample_idx = rng.choice(len(x_source), size=sample_size, replace=False)
    x_shap = x_source[sample_idx]

    explainer = shap.TreeExplainer(ngb, model_output=0)
    shap_values = explainer.shap_values(
        x_shap,
        approximate=approximate,
        check_additivity=not approximate,
    )
    shap.summary_plot(shap_values, x_shap, feature_names=feature_names)

#if enable_debug_plots:
#    _print_shap()


In [ ]:
num_samples = 1000

samples = _inverse_normalized_and_boxcox(y_vat_dist.sample(num_samples))

logging.info(f"Total non-finite (NaN or Inf) values: {np.sum(~np.isfinite(samples))}")

samples = np.clip(samples, a_min=4, a_max=30)

df_with_samples = validation_and_test_df.with_columns(
    pl.Series("samples", samples.T)
)
df_with_samples.tail(3)

In [ ]:
# Obtain the predictive distribution
y_forecast_year_dist = ngb.pred_dist(X_forecast_year, max_iter=ngb.best_val_loss_itr)

num_samples = 2000


# Sample from the predicted distribution in the Box-Cox space
forecast_samples = _inverse_normalized_and_boxcox(y_forecast_year_dist.sample(num_samples))

logging.info(f"Total non-finite (NaN or Inf) values: {np.sum(~np.isfinite(forecast_samples))}")
assert np.sum(~np.isfinite(forecast_samples)) == 0, "Found nan values in samples"


forecast_samples = np.clip(forecast_samples, a_min=4, a_max=30)

running_order_df = running_order_df.with_columns(
    pl.Series("pace_samples", forecast_samples.T).shrink_dtype()
)
# running_order_df.head(1)

In [ ]:
if enable_debug_plots:
    sns.histplot(y_forecast_year_dist.loc)


In [ ]:
if enable_debug_plots:
    sns.histplot(y_forecast_year_dist.scale)


In [ ]:

df_with_samples = df_with_samples.with_columns([
    pl.col('samples').arr.median().alias('pred_median'),
    #pl.col('samples').arr.to_list().list.mean().alias('pred_median'),
    
    pl.col('samples').arr.std().alias('pred_std'), # Does this make sense with paces?
    pl.col('samples').arr.to_list().list.eval(pl.element().quantile(0.05)).list.first().alias('pred_start'),
    pl.col('samples').arr.to_list().list.eval(pl.element().quantile(0.95)).list.first().alias('pred_end'),
])

df_with_samples = df_with_samples.with_columns([
    (pl.col('pace') - pl.col('pred_median')).alias('pred_error'),
    (pl.col('pace') - pl.col('pred_median')).abs().alias('pred_abs_error'),
])


running_order_df = running_order_df.with_columns([
    pl.col('pace_samples').arr.median().alias('pred_median'),
    #pl.col('pace_samples').arr.to_list().list.mean().alias('pred_median'),
    
    pl.col('pace_samples').arr.std().alias('pred_std'), # Does this make sense with paces?
    pl.col('pace_samples').arr.to_list().list.eval(pl.element().quantile(0.05)).list.first().alias('personal_start_95'),
    pl.col('pace_samples').arr.to_list().list.eval(pl.element().quantile(0.95)).list.first().alias('personal_end_95'),

    pl.col('pace_samples').arr.to_list().list.eval(pl.element().log()).list.median().alias('log_mean'),
    #pl.col('pace_samples').arr.to_list().list.eval(pl.element().log()).list.mean().alias('log_mean'),
    
    pl.col('pace_samples').arr.to_list().list.eval(pl.element().log()).list.std().alias('log_std'),
    # pl.col('pace_samples').arr.to_list().list.eval(pl.element().shrink_dtype()).alias('pace_samples'),
])

    
example_df = df_with_samples.filter(
    pl.col('unique_name').str.contains(example_name)
    #pl.col('unique_name').str.contains('jeppe koivula')
    #pl.col('unique_name').str.contains('janne ala-äijälä')
    #pl.col('unique_name').str.contains('timo rantalaiho')
    #pl.col('unique_name').str.contains('mikko mäkelä')
).select(['year', 'unique_name', 'pace', 'pred_median', 'pred_std', 'pred_start', 'pred_end', 'history_pace_median', 'run_num'])

example_df

In [ ]:
running_order_df.select(
    ['team_id', 'team', 'team_country', 'year', 'unique_name', 'pace', 'pred_median', 'pred_std', 'personal_start_95', 'personal_end_95', 'history_pace_median', 'run_num', 'pace_samples']
).filter(
    pl.col('team').str.contains(example_team)
)

In [ ]:
team_samples_df = running_order_df.filter(
    pl.col('team').str.contains(example_team)
).select(
    ['unique_name', 'pace',  'pace_samples']
).explode('pace_samples').rename({'pace_samples': 'pace_sample'}).with_columns(
    pl.col('pace_sample').cast(pl.Float32)
)

if enable_debug_plots:
    team_samples_df.describe()


In [ ]:
if enable_debug_plots and not team_samples_df.is_empty():
    plt.figure(figsize=(12, 8))
    ax = sns.kdeplot(team_samples_df, x='pace_sample', hue='unique_name', palette='bright')
    plt.title(race_id_str)
    #sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
forecasts_path = f'{results_dir}/running_order_samples_v2_{shared.race_id_str()}.json'
if not optuna_trial_number:
    running_order_df.write_json(forecasts_path)
    logging.info(f'Wrote {forecasts_path}')
else:
    logging.info(f'Skipping writing samples to {forecasts_path} during tuning')

In [ ]:
if enable_debug_plots:
    plt.figure(figsize=(14, 10))
    sns.set_style('whitegrid')

    sns.scatterplot(
        data=example_df,
        x='pace',
        y='pred_median',
        hue='run_num',
        size='pred_std',
        sizes=(20, 200),
        alpha=0.7,
        edgecolor=None,
        legend=True,
    )

    plt.errorbar(
        example_df['pace'],
        example_df['pred_median'],
        yerr=[example_df['pred_median'] - example_df['pred_start'], example_df['pred_end'] - example_df['pred_median']],
        fmt='none',
        ecolor='gray',
        alpha=0.5,
        capsize=3,
        label='95% Prediction Interval',
    )

    plt.plot(
        [example_df['pace'].min(), example_df['pace'].max()],
        [example_df['pace'].min(), example_df['pace'].max()],
        color='red',
        linestyle='--',
        linewidth=2,
        label='Perfect Prediction',
    )

    plt.xlabel('Actual Values')
    plt.ylabel('Predicted Mean')
    plt.title('Predictions with Uncertainty Visualization')
    plt.legend(loc='upper left')
    plt.show()


In [ ]:
if enable_debug_plots:
    plt.figure(figsize=(12, 8))

    plot_df = df_with_samples.filter(
        pl.col('pace') <= 30,
        pl.col('pred_median') <= 25,
    ).select(['year', 'pace', 'pred_median', 'pred_error', 'pred_abs_error', 'leg', 'run_num', 'team_country_truncated', 'history_pace_median', 'roll_5y_pace_mean', 'normalized_team_id', 'normalized_tc'])

    sns.scatterplot(data=plot_df, x='pred_median', y='pace', alpha=0.3, hue='year', palette='bright')
    plt.plot(
        [plot_df['pred_median'].min(), plot_df['pred_median'].max()],
        [plot_df['pred_median'].min(), plot_df['pred_median'].max()],
        color='red',
        linestyle='--',
        linewidth=2,
        label='Perfect Prediction',
    )

    plt.ylabel('True Pace')
    plt.title(race_id_str)
    plt.legend()
    plt.show()


In [ ]:
if enable_debug_plots:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=plot_df, x='pace', y='pred_error', alpha=0.3, hue='run_num', palette='bright')
    plt.title(f'Prediction Error Visualization {race_id_str}')


In [ ]:
if enable_debug_plots:
    plt.figure(figsize=(12, 8))
    sns.boxplot(data=plot_df, x='year', y='pred_abs_error', hue='year', palette='bright')
    plt.title(f'Prediction Error Visualization {race_id_str}')
